# Malaria Cell Classification — End-to-End ML Cycle
Binary image classification (**Parasitized** vs **Uninfected**) on the NIH malaria cell-image dataset.
Pipeline: data acquisition → preprocessing → model creation → testing → prediction → retraining.

Run this notebook from the `notebook/` folder (the first cell moves to the project root).

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import config
from src.preprocessing import (download_dataset, prepare_split, count_images, make_datasets,
                               dataset_features, sample_images, FEATURE_NAMES, load_image_array)
from src.model import (build_model, compile_model, train_model, save_model, evaluate_model,
                       load_trained_model)
from src.prediction import predict_image
print("Project root:", os.getcwd())

## 1. Data acquisition
Download from Kaggle (needs `kagglehub` + Kaggle credentials) **or** point `RAW_DIR` to an existing copy. `PER_CLASS` caps images per class (None = all ~13.7k per class); use a small number to keep the repo light.

In [ ]:
RAW_DIR = None          # e.g. "/content/cell_images" if you already downloaded the data
PER_CLASS = 2000        # None -> use every image
raw = RAW_DIR or download_dataset()
print(prepare_split(raw, per_class=PER_CLASS))
print("train:", count_images(config.TRAIN_DIR), "| test:", count_images(config.TEST_DIR))

## 2. Exploratory analysis & feature interpretation

In [ ]:
samples = sample_images(n=5)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for r, (cls, paths) in enumerate(samples.items()):
    for c, p in enumerate(paths):
        axes[r, c].imshow(plt.imread(p)); axes[r, c].axis("off")
        if c == 0: axes[r, c].set_title(cls, loc="left")
plt.tight_layout(); plt.show()

counts = pd.Series(count_images(config.TRAIN_DIR))
counts.plot.bar(title="Class balance (train)"); plt.show()

In [ ]:
df = dataset_features(per_class=300)
df.groupby("label")[FEATURE_NAMES].mean().T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, f in zip(axes.ravel(), FEATURE_NAMES):
    sns.boxplot(data=df, x="label", y=f, ax=ax); ax.set_title(f)
plt.tight_layout(); plt.show()

sns.heatmap(df[FEATURE_NAMES].corr(), annot=True, fmt=".2f", cmap="coolwarm"); plt.title("Feature correlation"); plt.show()

### Interpretation (fill in after running the cells above)
Write 1–2 sentences for **each of at least 3 features**: which class is higher, by how much, and what it means biologically
(e.g. *dark_stain_ratio* — stained parasites show up as dark purple regions inside the red blood cell).
- **Feature 1:** …
- **Feature 2:** …
- **Feature 3:** …

**Story:** …

## 3. Preprocessing
* Images resized to 128×128 RGB (`load_image_array`).
* Scaling to [-1, 1] and data augmentation (flips, rotation, zoom) are **layers inside the model**, so training and serving use identical preprocessing.
* 85/15 train/validation split (seeded) from `data/train`; `data/test` is held out for the final evaluation.

In [ ]:
train_ds, val_ds, test_ds = make_datasets()
for x, y in train_ds.take(1):
    print("batch:", x.shape, x.dtype, "| labels:", y.shape, "| value range:", float(x.numpy().min()), "-", float(x.numpy().max()))

## 4. Model creation & training
Transfer learning with **MobileNetV2** (ImageNet weights, frozen) + regularised head. Optimisation techniques used:
* **L2 regularisation** and **Dropout** in the classifier head
* **Adam** optimiser, **ReduceLROnPlateau** learning-rate schedule
* **EarlyStopping** (restores best weights)
* **Two-phase training**: train the head, then fine-tune the last 30 backbone layers at a low learning rate
* Data augmentation inside the model

To train a CNN from scratch instead, set `MODEL_TYPE = "custom_cnn"`.

In [ ]:
MODEL_TYPE = "mobilenet"     # or "custom_cnn"
model, history = train_model(train_ds, val_ds, epochs=8, fine_tune_epochs=4, model_type=MODEL_TYPE)
model.summary()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, k in zip(axes, ["loss", "accuracy"]):
    ax.plot(history[k], label="train"); ax.plot(history["val_" + k], label="val"); ax.set_title(k); ax.legend()
plt.show()

## 5. Model testing (held-out test set)

In [ ]:
metrics = evaluate_model(model, test_ds, history=history, extra={"model_type": MODEL_TYPE})
pd.Series({k: metrics[k] for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "loss"]})

In [ ]:
cm = np.array(metrics["confusion_matrix"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=metrics["classes"], yticklabels=metrics["classes"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix"); plt.show()
pd.DataFrame(metrics["report"]).T

**How good is it?** Write your reading of the results here: overall accuracy, recall on *Parasitized* (missed infections are the costly error in a screening setting), precision, F1 and ROC-AUC, and where the model makes mistakes.

## 6. Save the model & prediction function

In [ ]:
save_model(model)       # -> models/malaria_model.h5
print(config.MODEL_PATH, config.MODEL_PATH.stat().st_size // 1024, "KB")

In [ ]:
reloaded = load_trained_model()
def predict_from_path(path):
    """Predict one image file with the saved model."""
    return predict_image(path)

demo = sample_images(config.TEST_DIR, n=3)
for cls, paths in demo.items():
    for p in paths:
        print(f"actual={cls:12s} ->", predict_from_path(p))

## 7. Retraining
Retraining is implemented in `src/retrain.py` and exposed through the API / UI:
1. new labelled images are uploaded (`POST /upload`) and recorded in SQLite,
2. a trigger fires either manually (`POST /retrain`, UI button) or automatically when pending images ≥ `RETRAIN_THRESHOLD`,
3. images are merged into `data/train`, the current model is fine-tuned at a low learning rate, evaluated on the test set and hot-swapped.

Below: run the same logic directly from the notebook.

In [ ]:
from src import retrain
print("pending uploads:", retrain.pending_count(), "| trigger fires at:", config.RETRAIN_THRESHOLD)
# result = retrain.retrain(epochs=2)   # uncomment to run retraining here
# result